# Capstone Project Part 2
## Task 1 . Evaluate ViT Image Classification

In this section, we fine-tune a Vision Transformer (ViT) model on the beans dataset, evaluate its performance, and display the final metrics:
- Accuracy
- Precision
- Recall
- F1 Score

In [1]:
!pip install -q transformers datasets evaluate
!pip install -q --upgrade fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.5/202.5 kB 7.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
# Imports
from datasets import load_dataset
import numpy as np
import torch
import evaluate

from transformers import AutoImageProcessor

from datasets import load_dataset
import numpy as np
import torch
import evaluate

from transformers import (
    AutoImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

In [3]:
# Load dataset
ds = load_dataset("beans")
ds

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/18.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1034 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/133 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/128 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})

In [4]:
# Model checkpoint and image processor
model_name_or_path = "google/vit-base-patch16-224-in21k"
image_processor = AutoImageProcessor.from_pretrained(model_name_or_path)

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [5]:
# Transform dataset
def transform(example_batch):
    inputs = image_processor(
        [x for x in example_batch["image"]],
        return_tensors="pt"
    )
    inputs["labels"] = example_batch["labels"]
    return inputs

prepared_ds = ds.with_transform(transform)

In [6]:
# Data collator
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["labels"] for x in batch])
    }

In [7]:
# Load evaluation metrics
accuracy_metric = evaluate.load("accuracy")
precision_metric = evaluate.load("precision")
recall_metric = evaluate.load("recall")
f1_metric = evaluate.load("f1")

In [8]:
# Compute metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    precision = precision_metric.compute(predictions=predictions, references=labels, average="macro")["precision"]
    recall = recall_metric.compute(predictions=predictions, references=labels, average="macro")["recall"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [9]:
# Label mappings
labels = ds["train"].features["labels"].names

id2label = {i: label for i, label in enumerate(labels)}
label2id = {label: i for i, label in enumerate(labels)}

id2label, label2id

({0: 'angular_leaf_spot', 1: 'bean_rust', 2: 'healthy'},
 {'angular_leaf_spot': 0, 'bean_rust': 1, 'healthy': 2})

In [10]:
# Load pretrained ViT model
model = ViTForImageClassification.from_pretrained(
    model_name_or_path,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./vit-beans-results",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=4,
    learning_rate=2e-4,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_ds["train"],
    eval_dataset=prepared_ds["validation"],
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

In [13]:
# Train model
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.344920,0.143482,0.969925,0.972222,0.970202,0.970182
2,0.094548,0.091343,0.977444,0.979167,0.977273,0.977483
3,0.037590,0.068268,0.984962,0.985178,0.984848,0.984930
4,0.015442,0.057283,0.984962,0.985816,0.984848,0.985002


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
# Evaluate model
metrics = trainer.evaluate(prepared_ds["validation"])
metrics

{'eval_loss': 0.06826833635568619,
 'eval_accuracy': 0.9849624060150376,
 'eval_precision': 0.9851778656126483,
 'eval_recall': 0.9848484848484849,
 'eval_f1': 0.9849298211367178,
 'eval_runtime': 2.4125,
 'eval_samples_per_second': 55.13,
 'eval_steps_per_second': 3.731,
 'epoch': 4.0}

In [15]:
# Display final metrics
vit_accuracy = metrics["eval_accuracy"]
vit_precision = metrics["eval_precision"]
vit_recall = metrics["eval_recall"]
vit_f1 = metrics["eval_f1"]

print("ViT Final Metrics")
print(f"Accuracy : {vit_accuracy:.4f}")
print(f"Precision: {vit_precision:.4f}")
print(f"Recall   : {vit_recall:.4f}")
print(f"F1 Score : {vit_f1:.4f}")

ViT Final Metrics
Accuracy : 0.9850
Precision: 0.9852
Recall   : 0.9848
F1 Score : 0.9849


In [17]:
# Show results in tabular form
import pandas as pd
vit_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [vit_accuracy, vit_precision, vit_recall, vit_f1]
})

vit_results

,Metric,Value
0,Accuracy,0.984962
1,Precision,0.985178
2,Recall,0.984848
3,F1 Score,0.984930


## Observation

The Vision Transformer (ViT) achieved strong image classification performance, with good accuracy, precision, recall, and F1 score after fine-tuning. However, the training process is less efficient because it requires several epochs, GPU support, and more computation due to self-attention over image patches. Compared with simpler image models, ViT takes more time and resources to train, but this added cost improves classification quality. This shows a clear trade-off: ViT is slower and more computationally intensive during training, yet it produces accurate and robust output, making it a strong choice when model performance is more important than training speed.

In [18]:
# Save metrics for later comparison
vit_metrics_dict = {
    "accuracy": vit_accuracy,
    "precision": vit_precision,
    "recall": vit_recall,
    "f1": vit_f1
}

vit_metrics_dict

{'accuracy': 0.9849624060150376,
 'precision': 0.9851778656126483,
 'recall': 0.9848484848484849,
 'f1': 0.9849298211367178}

# Task 2 . Evaluate CLIP Image Classification

In this section, we evaluate CLIP as an image classifier without additional training.  
We use text prompts for each class and compare them with image embeddings to predict labels.

Final metrics displayed:
- Accuracy
- Precision
- Recall
- F1 Score

In [19]:
# Install required packages
!pip install -q transformers datasets evaluate pillow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 9.6 MB/s eta 0:00:00


In [20]:
# Imports
import torch
import numpy as np
import pandas as pd

from datasets import load_dataset
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [21]:
# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [22]:
# Load dataset
ds = load_dataset("beans")
ds

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})

In [23]:
# Show label names
label_names = ds["train"].features["labels"].names
label_names

['angular_leaf_spot', 'bean_rust', 'healthy']

In [24]:
# Load CLIP model and processor
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)
model.eval()

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [25]:
# Define text prompts for each class
# Keep prompts simple and aligned with class names
candidate_labels = label_names
text_prompts = [f"a photo of {label}" for label in candidate_labels]

print("Text prompts used for CLIP classification:")
text_prompts

Text prompts used for CLIP classification:


['a photo of angular_leaf_spot', 'a photo of bean_rust', 'a photo of healthy']

In [26]:
# Optional: inspect a sample image and label
sample = ds["validation"][0]
sample["image"], label_names[sample["labels"]]

(<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x500>,
 'angular_leaf_spot')

In [27]:
# CLIP zero-shot prediction function
def predict_clip(image, text_prompts, model, processor, device):
    inputs = processor(
        text=text_prompts,
        images=image,
        return_tensors="pt",
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    return pred_idx, probs

In [28]:
# Run prediction on validation set
y_true = []
y_pred = []

for example in ds["validation"]:
    image = example["image"]
    true_label = example["labels"]

    pred_label, probs = predict_clip(
        image=image,
        text_prompts=text_prompts,
        model=model,
        processor=processor,
        device=device
    )

    y_true.append(true_label)
    y_pred.append(pred_label)

print("Total evaluated samples:", len(y_true))

Total evaluated samples: 133


In [29]:
# Compute final metrics
clip_accuracy = accuracy_score(y_true, y_pred)
clip_precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
clip_recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
clip_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

print("CLIP Final Metrics")
print(f"Accuracy : {clip_accuracy:.4f}")
print(f"Precision: {clip_precision:.4f}")
print(f"Recall   : {clip_recall:.4f}")
print(f"F1 Score : {clip_f1:.4f}")

CLIP Final Metrics
Accuracy : 0.3308
Precision: 0.1103
Recall   : 0.3333
F1 Score : 0.1657


In [30]:
# Display metrics in table format
clip_results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Value": [clip_accuracy, clip_precision, clip_recall, clip_f1]
})

clip_results

,Metric,Value
0,Accuracy,0.330827
1,Precision,0.110276
2,Recall,0.333333
3,F1 Score,0.165725


In [31]:
# Optional: class-wise prediction preview for first few samples
preview_rows = []

for i, example in enumerate(ds["validation"].select(range(10))):
    image = example["image"]
    true_idx = example["labels"]
    pred_idx, probs = predict_clip(
        image=image,
        text_prompts=text_prompts,
        model=model,
        processor=processor,
        device=device
    )

    preview_rows.append({
        "Sample": i,
        "True Label": label_names[true_idx],
        "Predicted Label": label_names[pred_idx],
        "Confidence": float(np.max(probs))
    })

preview_df = pd.DataFrame(preview_rows)
preview_df

,Sample,True Label,Predicted Label,Confidence
0,0,angular_leaf_spot,angular_leaf_spot,0.930510
1,1,angular_leaf_spot,angular_leaf_spot,0.984091
2,2,angular_leaf_spot,angular_leaf_spot,0.904231
3,3,angular_leaf_spot,angular_leaf_spot,0.898068
4,4,angular_leaf_spot,angular_leaf_spot,0.936619
5,5,angular_leaf_spot,angular_leaf_spot,0.979752
6,6,angular_leaf_spot,angular_leaf_spot,0.968158
7,7,angular_leaf_spot,angular_leaf_spot,0.964064
8,8,angular_leaf_spot,angular_leaf_spot,0.982658
9,9,angular_leaf_spot,angular_leaf_spot,0.969789


## Observation

CLIP is highly efficient for image classification because it can be used directly without additional task-specific training. It relies on contrastive learning to connect images with text descriptions, making it fast and flexible for zero-shot classification. In this experiment, CLIP required much less computation and setup effort than fine-tuning a ViT model. However, its classification accuracy depends heavily on how well the text prompts match the dataset classes. As a result, CLIP is very efficient and practical for rapid classification tasks, but its accuracy may be lower than a supervised model that has been fine-tuned specifically on the dataset.

In [32]:
# Save metrics for later comparison
clip_metrics_dict = {
    "accuracy": clip_accuracy,
    "precision": clip_precision,
    "recall": clip_recall,
    "f1": clip_f1
}

clip_metrics_dict

{'accuracy': 0.3308270676691729,
 'precision': 0.11027568922305764,
 'recall': 0.3333333333333333,
 'f1': 0.1657250470809793}

# Task 3 . Compare and Contrast

In this section, we compare Vision Transformer (ViT) and CLIP for image classification with respect to:

- Training speed and efficiency
- Accuracy of output

In [35]:
# Check that metrics from Task 1 and Task 2 exist
print("ViT Metrics:", vit_metrics_dict)
print("CLIP Metrics:", clip_metrics_dict)

ViT Metrics: {'accuracy': 0.9849624060150376, 'precision': 0.9851778656126483, 'recall': 0.9848484848484849, 'f1': 0.9849298211367178}
CLIP Metrics: {'accuracy': 0.3308270676691729, 'precision': 0.11027568922305764, 'recall': 0.3333333333333333, 'f1': 0.1657250470809793}


In [36]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Model": ["ViT", "CLIP"],
    "Accuracy": [vit_metrics_dict["accuracy"], clip_metrics_dict["accuracy"]],
    "Precision": [vit_metrics_dict["precision"], clip_metrics_dict["precision"]],
    "Recall": [vit_metrics_dict["recall"], clip_metrics_dict["recall"]],
    "F1 Score": [vit_metrics_dict["f1"], clip_metrics_dict["f1"]],
    "Training Required": ["Yes, fine-tuning required", "No additional training required"],
    "Efficiency": [
        "Lower efficiency during setup and training",
        "Higher efficiency for immediate use"
    ]
})

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,Training Required,Efficiency
0,ViT,0.984962,0.985178,0.984848,0.984930,"Yes, fine-tuning required",Lower efficiency during setup and training
1,CLIP,0.330827,0.110276,0.333333,0.165725,No additional training required,Higher efficiency for immediate use


In [37]:
# Round values for cleaner display
comparison_df_rounded = comparison_df.copy()

for col in ["Accuracy", "Precision", "Recall", "F1 Score"]:
    comparison_df_rounded[col] = comparison_df_rounded[col].round(4)

comparison_df_rounded

,Model,Accuracy,Precision,Recall,F1 Score,Training Required,Efficiency
0,ViT,0.9850,0.9852,0.9848,0.9849,"Yes, fine-tuning required",Lower efficiency during setup and training
1,CLIP,0.3308,0.1103,0.3333,0.1657,No additional training required,Higher efficiency for immediate use


In [38]:
# Simple summary statements
if vit_metrics_dict["accuracy"] > clip_metrics_dict["accuracy"]:
    better_accuracy_model = "ViT"
else:
    better_accuracy_model = "CLIP"

print(f"Model with higher accuracy: {better_accuracy_model}")
print("ViT requires fine-tuning before evaluation.")
print("CLIP can be used directly in zero-shot mode without additional training.")

Model with higher accuracy: ViT
ViT requires fine-tuning before evaluation.
CLIP can be used directly in zero-shot mode without additional training.


## Comparison

The two models show a clear trade-off between efficiency and classification accuracy. ViT requires fine-tuning on the dataset before it can produce results, which makes it slower and more computationally expensive. Training involves multiple epochs, more GPU usage, and longer execution time. However, this extra effort improves task-specific learning and usually leads to stronger classification performance. In this experiment, ViT achieved higher accuracy-related metrics after supervised training, showing that it is better suited when classification quality is the main priority.

CLIP, in contrast, is much faster and more efficient to use because it does not need additional training. It performs classification by matching images with text prompts using contrastive learning. This makes it highly practical for zero-shot image classification and quick experiments. However, its accuracy depends on how well the text prompts represent the image classes, so its performance may be lower than a fine-tuned ViT model. Overall, ViT is better when high accuracy is needed, while CLIP is better when speed, flexibility, and minimal setup are more important.

In [40]:
# Export comparison table if needed
comparison_df_rounded.to_csv("vit_clip_comparison.csv", index=False)
print("Saved comparison table as vit_clip_comparison.csv")

Saved comparison table as vit_clip_comparison.csv


## Final Conclusion

- **ViT**
  - Better task-specific performance after fine-tuning
  - More training time and computational cost

- **CLIP**
  - No additional training needed
  - Faster and more efficient for immediate use
  - Accuracy depends on the quality of text prompts

This shows that the best model choice depends on the objective:
- Choose **ViT** when classification accuracy is the priority
- Choose **CLIP** when speed, efficiency, and zero-shot capability are more important